In [80]:
import pandas as pd 
from pathlib import Path
import os

# 理解每張表的grain

In [81]:
os.chdir("/Users/kaiping/Desktop/olist_project/data") 

In [82]:
os.getcwd()

'/Users/kaiping/Desktop/olist_project/data'

In [83]:
df_translation = pd.read_csv("raw/product_category_name_translation.csv")
df_sellers     = pd.read_csv("raw/olist_sellers_dataset.csv")
df_products    = pd.read_csv("raw/olist_products_dataset.csv")
df_orders      = pd.read_csv("raw/olist_orders_dataset.csv")
df_order_reviews  = pd.read_csv("raw/olist_order_reviews_dataset.csv")
df_order_payments = pd.read_csv("raw/olist_order_payments_dataset.csv")
df_order_items    = pd.read_csv("raw/olist_order_items_dataset.csv")
df_geolocation    = pd.read_csv("raw/olist_geolocation_dataset.csv")
df_customers      = pd.read_csv("raw/olist_customers_dataset.csv")


In [84]:
# 把df收進去 dict
tables = {
    "translation": df_translation,
    "sellers": df_sellers,
    "products": df_products,
    "orders": df_orders,
    "order_reviews": df_order_reviews,
    "order_payments": df_order_payments,
    "order_items": df_order_items,
    "geolocation": df_geolocation,
    "customers": df_customers,
}


In [85]:
grain_df = pd.DataFrame([
    {
        "table": "olist_orders_dataset",
        "grain": "一列代表一張訂單（訂單層級事件，包含狀態與時間節點）"
    },
    {
        "table": "olist_order_items_dataset",
        "grain": "一列代表一張訂單中的一筆商品明細（訂單 × 商品 × 賣家）"
    },
    {
        "table": "olist_order_payments_dataset",
        "grain": "一列代表一筆付款事件（同一訂單可能有多筆付款或分期）"
    },
    {
        "table": "olist_order_reviews_dataset",
        "grain": "一列代表一則訂單評論（以評論為單位，而非訂單）"
    },
    {
        "table": "olist_customers_dataset",
        "grain": "一列代表一次下單時的顧客快照(客戶每下一筆訂單就會產生一筆顧客資料快照)"
    },
    {
        "table": "olist_products_dataset",
        "grain": "一列代表一個商品（商品維度資料）"
    },
    {
        "table": "olist_sellers_dataset",
        "grain": "一列代表一個賣家（賣家維度資料）"
    },
    {
        "table": "olist_geolocation_dataset",
        "grain": "一列代表一筆地理位置記錄（郵遞區號前綴 × 經緯度，可能重複）"
    },
    {
        "table": "product_category_name_translation",
        "grain": "一列代表一個商品品類名稱的語言翻譯對照"
    }
])

grain_df


,table,grain
0,olist_orders_dataset,一列代表一張訂單（訂單層級事件，包含狀態與時間節點）
1,olist_order_items_dataset,一列代表一張訂單中的一筆商品明細（訂單 × 商品 × 賣家）
2,olist_order_payments_dataset,一列代表一筆付款事件（同一訂單可能有多筆付款或分期）
3,olist_order_reviews_dataset,一列代表一則訂單評論（以評論為單位，而非訂單）
4,olist_customers_dataset,一列代表一次下單時的顧客快照(客戶每下一筆訂單就會產生一筆顧客資料快照)
5,olist_products_dataset,一列代表一個商品（商品維度資料）
6,olist_sellers_dataset,一列代表一個賣家（賣家維度資料）
7,olist_geolocation_dataset,一列代表一筆地理位置記錄（郵遞區號前綴 × 經緯度，可能重複）
8,product_category_name_translation,一列代表一個商品品類名稱的語言翻譯對照


In [86]:
os.chdir("/Users/kaiping/Desktop/olist_project/notebooks/data_understanding")
output_path = Path("grain.csv") # 只給檔名 相對路徑
if not output_path.exists():
    grain_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"已儲存：{output_path}（位於 {os.getcwd()}）")
else:
    print(f"已存在，未覆蓋：{output_path}（位於 {os.getcwd()}）")

已儲存：grain.csv（位於 /Users/kaiping/Desktop/olist_project/notebooks/data_understanding）


# 驗證Pk唯一性

In [87]:
tables_pk = {
    "orders": (df_orders, "order_id"),
    "order_items": (df_order_items, ["order_id", "order_item_id"]),
    "payments": (df_order_payments, ["order_id", "payment_sequential"]),
    "reviews": (df_order_reviews, "review_id"),
    "customers": (df_customers, "customer_id"),
    "products": (df_products, "product_id"),
    "sellers": (df_sellers, "seller_id"),
}
# 定義清楚每張表的候選PK，value是一個tuple

In [74]:
for name, (df, pk) in tables_pk.items(): # for迴圈dict
#  "orders": (df_orders, "order_id")，df,pk接tuple變數
    print(f"\n=== {name} ===")

    # 單一 PK 如果是字串就是只有PK 
    if isinstance(pk, str):
        print("Primary Key:",pk)
        print("NULL:", df[pk].isna().sum())
        print("COUNT:", df[pk].count())
        print("NUNIQUE:", df[pk].nunique())

    # 複合 PK
    else:
        print("Primary Key:",pk)
        print("Pk_count:",df[pk].dropna().shape[0])
        print("NULL:", df[pk].isna().any(axis=1).sum()) # any(axis=1): 2維df 如果一row為Ture就為True
        print("DUPLICATED:", df[pk].duplicated().sum()) # 檢查復合主鍵組合是否有重複



=== orders ===
Primary Key: order_id
NULL: 0
COUNT: 99441
NUNIQUE: 99441

=== order_items ===
Primary Key: ['order_id', 'order_item_id']
Pk_count: 112650
NULL: 0
DUPLICATED: 0

=== payments ===
Primary Key: ['order_id', 'payment_sequential']
Pk_count: 103886
NULL: 0
DUPLICATED: 0

=== reviews ===
Primary Key: review_id
NULL: 0
COUNT: 99224
NUNIQUE: 98410

=== customers ===
Primary Key: customer_id
NULL: 0
COUNT: 99441
NUNIQUE: 99441

=== products ===
Primary Key: product_id
NULL: 0
COUNT: 32951
NUNIQUE: 32951

=== sellers ===
Primary Key: seller_id
NULL: 0
COUNT: 3095
NUNIQUE: 3095


**review表的PK沒有唯一**  
代表重複數量 = COUNT - NUNIQUE = 99224 - 98410 = 814筆

# 驗證Pk唯一性


In [94]:
fk_rules = pd.DataFrame([
    # orders -> customers
    {"child_table": "orders", "child_fk_cols": "customer_id",
     "parent_table": "customers", "parent_pk_cols": "customer_id"},

    # order_items -> orders/products/sellers
    {"child_table": "order_items", "child_fk_cols": "order_id",
     "parent_table": "orders", "parent_pk_cols": "order_id"},
    {"child_table": "order_items", "child_fk_cols": "product_id",
     "parent_table": "products", "parent_pk_cols": "product_id"},
    {"child_table": "order_items", "child_fk_cols": "seller_id",
     "parent_table": "sellers", "parent_pk_cols": "seller_id"},

    # order_payments -> orders
    {"child_table": "order_payments", "child_fk_cols": "order_id",
     "parent_table": "orders", "parent_pk_cols": "order_id"},

    # order_reviews -> orders
    {"child_table": "order_reviews", "child_fk_cols": "order_id",
     "parent_table": "orders", "parent_pk_cols": "order_id"},

    # products -> translation (category)
    {"child_table": "products", "child_fk_cols": "product_category_name",
     "parent_table": "translation", "parent_pk_cols": "product_category_name"},
])


In [95]:
fk_rules

,child_table,child_fk_cols,parent_table,parent_pk_cols
0,orders,customer_id,customers,customer_id
1,order_items,order_id,orders,order_id
2,order_items,product_id,products,product_id
3,order_items,seller_id,sellers,seller_id
4,order_payments,order_id,orders,order_id
5,order_reviews,order_id,orders,order_id
6,products,product_category_name,translation,product_category_name


In [96]:

def _split_cols(s: str) -> list[str]:
    return [c.strip() for c in str(s).split("|") if c.strip()]
# 先把字串用 | 分割成 list，對每個元素去掉前後空白，如果去完後不是空字串，就把這個結果收進新的 list 回傳。
# for c in str(s).split("|"):
    # cleaned = c.strip()      
    # if cleaned:
        #result.append(cleaned)

def validate_fk_rules(fk_rules: pd.DataFrame, tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    out_rows = []

    for _, r in fk_rules.iterrows():
        child_table = r["child_table"]
        parent_table = r["parent_table"]
        child_fk_cols = _split_cols(r["child_fk_cols"])
        parent_pk_cols = _split_cols(r["parent_pk_cols"])

        # 取表
        if child_table not in tables:
            raise KeyError(f"child_table '{child_table}' not found in tables dict")
        if parent_table not in tables:
            raise KeyError(f"parent_table '{parent_table}' not found in tables dict")

        child = tables[child_table]
        parent = tables[parent_table]

        # 欄位存在性檢查
        missing_child_cols = [c for c in child_fk_cols if c not in child.columns]
        missing_parent_cols = [c for c in parent_pk_cols if c not in parent.columns]
        if missing_child_cols:
            raise KeyError(f"{child_table} missing columns: {missing_child_cols}")
        if missing_parent_cols:
            raise KeyError(f"{parent_table} missing columns: {missing_parent_cols}")

        # parent row count
        parent_row_cnt = int(len(parent))

        # FK 非空（複合鍵：整組都非空才算 non-null）
        child_fk_df = child[child_fk_cols]
        fk_nonnull_mask = child_fk_df.notna().all(axis=1)

        fk_null_cnt = int((~fk_nonnull_mask).sum())
        fk_null_rate = float(fk_null_cnt / len(child)) if len(child) > 0 else 0.0

        child_fk_nonnull = child_fk_df.loc[fk_nonnull_mask].copy()

        # child_fk_distinct_cnt（複合鍵用 drop_duplicates）
        child_fk_distinct_cnt = int(child_fk_nonnull.drop_duplicates().shape[0])

        # 找 missing FK：只用 FK 非空的列去對 parent key
        parent_keys_df = parent[parent_pk_cols].drop_duplicates()

        merged = child_fk_nonnull.merge(
            parent_keys_df,
            how="left",
            left_on=child_fk_cols,
            right_on=parent_pk_cols,
            indicator=True
        )
        missing_fk_cnt = int((merged["_merge"] == "left_only").sum())

        denom = len(child_fk_nonnull)  # 分母 = FK 非空筆數
        missing_fk_rate = float(missing_fk_cnt / denom) if denom > 0 else 0.0

        # join cardinality 驗證：many_to_one（child→parent）
        # 目的：確保 parent join key 在 parent 端是唯一，避免 join 倍增
        try:
            _ = child.merge(
                parent,
                how="left",
                left_on=child_fk_cols,
                right_on=parent_pk_cols,
                validate="many_to_one"
            )
            validate_ok = True
        except Exception:
            validate_ok = False

        # A 版：validate_pass 只看「join 型態 OK」且「missing FK = 0」
        #      不把 FK NULL 視為 fail（但會在報表欄位顯示出來）
        validate_pass = bool(validate_ok and (missing_fk_cnt == 0))

        out_rows.append({
            "child_table": child_table,
            "child_fk_cols": "|".join(child_fk_cols),
            "parent_table": parent_table,
            "parent_pk_cols": "|".join(parent_pk_cols),

            # 新增：FK NULL 指標
            "fk_null_cnt": fk_null_cnt,
            "fk_null_rate": fk_null_rate,

            # 原本：missing FK 指標
            "missing_fk_cnt": missing_fk_cnt,
            "missing_fk_rate": missing_fk_rate,

            # 其他輔助指標
            "parent_row_cnt": parent_row_cnt,
            "child_fk_distinct_cnt": child_fk_distinct_cnt,

            # 結果
            "validate_pass": validate_pass
        })

    return pd.DataFrame(out_rows)




In [97]:
fk_report = validate_fk_rules(fk_rules, tables)
fk_report.sort_values(["validate_pass", "missing_fk_cnt"], ascending=[True, False])


,child_table,child_fk_cols,parent_table,parent_pk_cols,fk_null_cnt,fk_null_rate,missing_fk_cnt,missing_fk_rate,parent_row_cnt,child_fk_distinct_cnt,validate_pass
6,products,product_category_name,translation,product_category_name,610,0.018512,13,0.000402,71,73,False
0,orders,customer_id,customers,customer_id,0,0.000000,0,0.000000,99441,99441,True
1,order_items,order_id,orders,order_id,0,0.000000,0,0.000000,99441,98666,True
2,order_items,product_id,products,product_id,0,0.000000,0,0.000000,32951,32951,True
3,order_items,seller_id,sellers,seller_id,0,0.000000,0,0.000000,3095,3095,True
4,order_payments,order_id,orders,order_id,0,0.000000,0,0.000000,99441,99440,True
5,order_reviews,order_id,orders,order_id,0,0.000000,0,0.000000,99441,98673,True


在 products 表中（FK 非空），有 13 筆產品 的 product_category_name
在 translation 表（71 筆 key）裡完全找不到

In [101]:
os.chdir("/Users/kaiping/Desktop/olist_project/notebooks/data_understanding")
fk_check_output_path = Path("fk_check.csv") # 只給檔名 相對路徑
if not fk_check_output_path.exists():
    fk_report.to_csv(fk_check_output_path, index=False, encoding="utf-8-sig")
    print(f"已儲存：{fk_check_output_path}（位於 {os.getcwd()}）")
else:
    print(f"已存在，未覆蓋：{fk_check_output_path}（位於 {os.getcwd()}）")

已儲存：fk_check.csv（位於 /Users/kaiping/Desktop/olist_project/notebooks/data_understanding）
